# CMB 距离先验与强透镜似然

本教程演示 HIcosmo 的 CMB 和强透镜似然函数。

**关键 API**：
- `Planck(model)` — Planck 2018 CMB 距离先验
- `H0LiCOW(model)` — H0LiCOW 强透镜时间延迟
- `TDCOSMO(model)` — TDCOSMO 层次贝叶斯分析
- `SH0ESLikelihood()` — SH0ES 距离阶梯
- `hicosmo('LCDM', 'planck', ...)` — 极简推断 API

In [ ]:
import hicosmo as hc
hc.init()

## 1. Planck 2018 CMB 距离先验

使用压缩参数 $(l_A, R, \omega_b, n_s)$ 约束宇宙学。

In [ ]:
from hicosmo.likelihoods import Planck
from hicosmo.models import LCDM

cmb = Planck(LCDM)

print(f"log L(Planck) = {cmb(H0=67.36, Omega_m=0.3153, Omega_b=0.0493):.2f}")
print(f"log L(H0=73)  = {cmb(H0=73, Omega_m=0.3, Omega_b=0.05):.2f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# CMB 对 H0 的约束
H0_grid = np.linspace(60, 80, 100)
L_cmb = np.array([cmb(H0=H0, Omega_m=0.3153, Omega_b=0.0493) for H0 in H0_grid])

plt.figure(figsize=(10, 5))
plt.plot(H0_grid, np.exp(L_cmb - L_cmb.max()), 'b-', lw=2, label='Planck 2018')
plt.axvline(67.36, c='red', ls='--', label='Best-fit')
plt.xlabel(r'$H_0$ [km/s/Mpc]'); plt.ylabel('Likelihood'); plt.legend()
plt.title('Planck CMB 距离先验')
plt.savefig('figures/05_cmb_h0.pdf', dpi=150, bbox_inches='tight')
plt.show()

## 2. H0LiCOW 强透镜

时间延迟距离：$D_{\Delta t} = (1+z_l) D_l D_s / D_{ls}$

In [ ]:
from hicosmo.likelihoods import H0LiCOW

h0licow = H0LiCOW(LCDM)

print(f"透镜系统数: 7")
print(f"log L(H0=73.3) = {h0licow(H0=73.3, Omega_m=0.3):.2f}")

In [ ]:
# H0LiCOW 对 H0 的约束
L_h0l = np.array([h0licow(H0=H0, Omega_m=0.3) for H0 in H0_grid])

plt.figure(figsize=(10, 5))
plt.plot(H0_grid, np.exp(L_h0l - L_h0l.max()), 'g-', lw=2, label='H0LiCOW')
plt.axvline(73.3, c='green', ls='--', label='Best-fit')
plt.axvline(67.36, c='blue', ls=':', label='Planck')
plt.xlabel(r'$H_0$ [km/s/Mpc]'); plt.ylabel('Likelihood'); plt.legend()
plt.title('H0LiCOW 强透镜时间延迟')
plt.savefig('figures/05_h0licow.pdf', dpi=150, bbox_inches='tight')
plt.show()

## 3. TDCOSMO 层次贝叶斯

包含质量片转换 (MST) 和恒星各向异性等系统效应。

In [ ]:
from hicosmo.likelihoods import TDCOSMO

tdcosmo = TDCOSMO(LCDM)

print("nuisance 参数:")
for p in tdcosmo.nuisance_parameters():
    print(f"  {p.name}: {p.value} ({p.prior['min']}, {p.prior['max']})")

In [ ]:
# 计算似然（需要 nuisance 参数）
log_L = tdcosmo(
    H0=74.0, Omega_m=0.3,
    lambda_int_mean=1.0, lambda_int_sigma=0.05,
    a_ani_mean=1.0, a_ani_sigma=0.1
)
print(f"log L(TDCOSMO) = {log_L:.2f}")

## 4. SH0ES 距离阶梯

In [ ]:
from hicosmo.likelihoods import SH0ESLikelihood

shoes = SH0ESLikelihood()
print("SH0ES: H0 = 73.04 ± 1.04 km/s/Mpc")

# SH0ES 似然
L_shoes = np.array([shoes.log_likelihood(LCDM(H0=H0, Omega_m=0.3)) for H0 in H0_grid])

plt.figure(figsize=(10, 5))
plt.plot(H0_grid, np.exp(L_shoes - L_shoes.max()), 'orange', lw=2, label='SH0ES')
plt.axvline(73.04, c='orange', ls='--', label='Best-fit')
plt.xlabel(r'$H_0$ [km/s/Mpc]'); plt.ylabel('Likelihood'); plt.legend()
plt.title('SH0ES 距离阶梯')
plt.savefig('figures/05_shoes.pdf', dpi=150, bbox_inches='tight')
plt.show()

## 5. Hubble Tension 可视化

In [ ]:
# 多探针 H0 约束比较
H0_grid = np.linspace(62, 80, 200)
L_cmb = [cmb(H0=H0, Omega_m=0.3153, Omega_b=0.0493) for H0 in H0_grid]
L_h0l = [h0licow(H0=H0, Omega_m=0.3) for H0 in H0_grid]
L_shoes = [shoes.log_likelihood(LCDM(H0=H0, Omega_m=0.3)) for H0 in H0_grid]

fig, ax = plt.subplots(figsize=(12, 6))

L_cmb = np.array(L_cmb); ax.fill_between(H0_grid, np.exp(L_cmb - L_cmb.max()), alpha=0.3, color='blue')
ax.plot(H0_grid, np.exp(L_cmb - L_cmb.max()), 'b-', lw=2, label='Planck CMB')

L_h0l = np.array(L_h0l); ax.fill_between(H0_grid, np.exp(L_h0l - L_h0l.max()), alpha=0.3, color='green')
ax.plot(H0_grid, np.exp(L_h0l - L_h0l.max()), 'g-', lw=2, label='H0LiCOW')

L_shoes = np.array(L_shoes); ax.fill_between(H0_grid, np.exp(L_shoes - L_shoes.max()), alpha=0.3, color='orange')
ax.plot(H0_grid, np.exp(L_shoes - L_shoes.max()), 'orange', lw=2, label='SH0ES')

ax.axvline(67.36, c='blue', ls=':'); ax.axvline(73.04, c='orange', ls=':')
ax.set_xlabel(r'$H_0$ [km/s/Mpc]'); ax.set_ylabel('Likelihood')
ax.set_title('Hubble Tension: 多探针 H0 约束'); ax.legend()

plt.tight_layout()
plt.savefig('figures/05_hubble_tension.pdf', dpi=150, bbox_inches='tight')
plt.show()

## 6. MCMC 参数约束

In [ ]:
from hicosmo import hicosmo

# 极简 API
inf = hicosmo('LCDM', 'planck', ['H0', 'Omega_m'])
samples = inf.run(num_samples=2000)
inf.summary()

In [ ]:
inf.corner_plot('figures/05_planck_corner.pdf')

In [ ]:
# H0LiCOW 约束
inf_h0l = hicosmo('LCDM', 'h0licow', ['H0', 'Omega_m'])
inf_h0l.run(num_samples=2000)
inf_h0l.summary()

## 7. 联合约束（SN + BAO + CMB）

In [ ]:
# 多探针联合
inf_joint = hicosmo('LCDM', ['sn', 'bao', 'planck'], ['H0', 'Omega_m'])
samples = inf_joint.run(num_samples=2000)
inf_joint.summary()

In [ ]:
inf_joint.corner_plot('figures/05_joint_corner.pdf')

## 8. 对象 API（更多控制）

In [ ]:
from hicosmo.samplers import MCMC
from hicosmo.likelihoods import SN_likelihood, BAO_likelihood, Planck
from hicosmo.visualization import Plotter

# 自定义似然组合
sne = SN_likelihood(LCDM, "pantheon+")
bao = BAO_likelihood(LCDM, "desi2024")
cmb = Planck(LCDM)

params = {
    'H0': (68.0, 60.0, 80.0),
    'Omega_m': (0.3, 0.1, 0.5),
}

# 联合似然用 + 运算符
mcmc = MCMC(params, sne + bao + cmb, chain_name='cmb_custom')
mcmc.run(num_samples=2000)
mcmc.print_summary()

In [ ]:
plotter = Plotter('cmb_custom')
plotter.corner(['H0', 'Omega_m'])
plotter.report()

## API 速查

```python
from hicosmo import hicosmo, list_likelihoods
from hicosmo.likelihoods import Planck, H0LiCOW, TDCOSMO, SH0ESLikelihood

# 创建似然
cmb = Planck(LCDM)                        # Planck 2018 距离先验
h0licow = H0LiCOW(LCDM)                   # H0LiCOW 强透镜
tdcosmo = TDCOSMO(LCDM)                   # TDCOSMO 层次贝叶斯
shoes = SH0ESLikelihood()                 # SH0ES 距离阶梯

# 计算似然
log_L = cmb(H0=67.36, Omega_m=0.3153, Omega_b=0.0493)
log_L = h0licow(H0=73.3, Omega_m=0.3)
log_L = tdcosmo(H0=74, Omega_m=0.3, lambda_int_mean=1.0, ...)
log_L = shoes.log_likelihood(LCDM(H0=73, Omega_m=0.3))

# TDCOSMO nuisance 参数
tdcosmo.nuisance_parameters()  # lambda_int_mean, lambda_int_sigma, a_ani_mean, ...

# 似然组合
joint = sne + bao + cmb + h0licow

# 极简推断 API
inf = hicosmo('LCDM', 'planck', ['H0', 'Omega_m'])           # CMB only
inf = hicosmo('LCDM', 'h0licow', ['H0', 'Omega_m'])          # H0LiCOW
inf = hicosmo('LCDM', ['sn', 'bao', 'planck'], ['H0', 'Omega_m'])  # 联合
inf.run(num_samples=5000)
inf.summary()
inf.corner_plot('output.pdf')
```

### H0 测量汇总

| 探针 | H0 [km/s/Mpc] | 类型 |
|------|---------------|------|
| Planck CMB | 67.36 ± 0.54 | 早期宇宙 |
| H0LiCOW | 73.3 +1.7/-1.8 | 强透镜 |
| SH0ES | 73.04 ± 1.04 | 距离阶梯 |
| TDCOSMO | ~74 | 层次贝叶斯 |